# Volume Analysis Notebook

Interactive analysis of 3D reciprocal space volumes (.h5 files).
Extends rsp_analysis.ipynb with volume slicing, isosurface, and custom analysis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LogNorm
%matplotlib inline

from volume_builder import load_volume_h5, compute_plane_M_inv
from rsp_reader import read_rsp_layer

## 1. Load Volume

In [ ]:
vol = load_volume_h5('path/to/volume_sym_mbar3m.h5')

print(f'Shape: {vol.intensity.shape}')
print(f'H: [{vol.H[0]:.3f}, {vol.H[-1]:.3f}] ({len(vol.H)} pts)')
print(f'K: [{vol.K[0]:.3f}, {vol.K[-1]:.3f}] ({len(vol.K)} pts)')
print(f'L: [{vol.L[0]:.3f}, {vol.L[-1]:.3f}] ({len(vol.L)} pts)')

cell = vol.metadata.get('cell')
if cell:
    print(f'Cell: a={cell["a"]:.4f} b={cell["b"]:.4f} c={cell["c"]:.4f}')
    print(f'      alpha={cell["alpha"]:.2f} beta={cell["beta"]:.2f} gamma={cell["gamma"]:.2f}')

## 2. HK Slice with Grid Overlay

In [ ]:
def plot_hk_slice(vol, l_value=0, vmax=None, cmap='gnuplot2', r=6):
    """Plot an HK slice from the volume with tilted grid overlay."""
    il = np.argmin(np.abs(vol.L - l_value))
    sl = vol.intensity[:, :, il].T.astype(float)  # (nk, nh) for imshow
    
    M_inv = compute_plane_M_inv(vol.metadata['ub'], vol.metadata['wavelength'], 'HK')
    s = vol.metadata.get('s', 0.001702)
    bin_xy = vol.metadata.get('bin_xy', 1)
    s_eff = s * bin_xy
    ny, nx = sl.shape
    cx = (nx + 1) / 2.0
    cy = (ny + 1) / 2.0
    extent = [(0.5-cx)*s_eff, (nx+0.5-cx)*s_eff, (0.5-cy)*s_eff, (ny+0.5-cy)*s_eff]
    
    if vmax is None:
        nz = sl[sl > 0]
        vmax = np.percentile(nz, 99) if len(nz) > 0 else 1
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(sl, extent=extent, origin='lower', aspect='equal',
              cmap=cmap, norm=Normalize(vmin=0, vmax=vmax))
    
    # Tilted grid
    M = np.linalg.inv(M_inv)
    corners = [(-r,-r),(r,-r),(-r,r),(r,r)]
    ax.set_xlim(min(M[0,0]*a+M[0,1]*b for a,b in corners),
                max(M[0,0]*a+M[0,1]*b for a,b in corners))
    ax.set_ylim(min(M[1,0]*a+M[1,1]*b for a,b in corners),
                max(M[1,0]*a+M[1,1]*b for a,b in corners))
    xlim = np.array(ax.get_xlim())
    for n in range(-r, r+1):
        if abs(M_inv[0,1]) > 1e-12:
            y0 = (n - M_inv[0,0]*xlim[0]) / M_inv[0,1]
            y1 = (n - M_inv[0,0]*xlim[1]) / M_inv[0,1]
            ax.plot(xlim, [y0, y1], 'w-', alpha=0.3, lw=0.5)
        else:
            ax.axvline(n / M_inv[0,0], color='w', alpha=0.3, lw=0.5)
        if abs(M_inv[1,0]) > 1e-12:
            y0 = (n - M_inv[1,0]*xlim[0]) / M_inv[1,1]
            y1 = (n - M_inv[1,0]*xlim[1]) / M_inv[1,1]
            ax.plot(xlim, [y0, y1], 'w-', alpha=0.3, lw=0.5)
        else:
            ax.axhline(n / M_inv[1,1], color='w', alpha=0.3, lw=0.5)
    
    ax.set_xlabel('h'); ax.set_ylabel('k')
    ax.set_title(f'l = {vol.L[il]:.3f}')
    return fig, ax

plot_hk_slice(vol, l_value=0, r=6);

## 3. 1D Line Profiles

In [ ]:
def line_profile_h(vol, k_value, l_value):
    """Extract 1D intensity profile along h at fixed k, l."""
    ik = np.argmin(np.abs(vol.K - k_value))
    il = np.argmin(np.abs(vol.L - l_value))
    return vol.H, vol.intensity[:, ik, il].astype(float)

def line_profile_k(vol, h_value, l_value):
    """Extract 1D intensity profile along k at fixed h, l."""
    ih = np.argmin(np.abs(vol.H - h_value))
    il = np.argmin(np.abs(vol.L - l_value))
    return vol.K, vol.intensity[ih, :, il].astype(float)

# Example: h-scan at k=0, l=0
h, I = line_profile_h(vol, k_value=0, l_value=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(h, I, 'b-', lw=0.8)
ax.set_xlabel('h'); ax.set_ylabel('Intensity')
ax.set_title('h-scan at k=0, l=0')
ax.set_xlim(-6, 6);

## 4. Multiple Slices Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
l_values = [0, 0.5, 1.0, 1.5, 2.0, 2.5]

for ax, l_val in zip(axes.flat, l_values):
    il = np.argmin(np.abs(vol.L - l_val))
    sl = vol.intensity[:, :, il].T.astype(float)
    nz = sl[sl > 0]
    vmax = np.percentile(nz, 99) if len(nz) > 0 else 1
    ax.imshow(sl, origin='lower', aspect='auto', cmap='gnuplot2',
              norm=Normalize(vmin=0, vmax=vmax))
    ax.set_title(f'l = {vol.L[il]:.2f}')

fig.suptitle('HK slices at different l values', fontsize=14)
fig.tight_layout();

## 5. Isosurface (3D)

In [ ]:
from volume_isosurface import plot_isosurface_notebook

# Crop to a small region for performance
fig = plot_isosurface_notebook(
    vol, isovalue=50,
    h_range=(-5, 5), k_range=(-5, 5), l_range=(-3, 3),
    colormap='plasma', opacity=0.5)
fig.show()

## 6. Compare with Single .img File

In [ ]:
# Load a single .img for comparison
layer = read_rsp_layer('path/to/single_frame.img')

print(f'Plane: {layer.plane_type}, {layer.fixed_label} = {layer.fixed_value}')
print(f'Shape: {layer.intensity.shape}')
print(f'{layer.x_label}: [{layer.idx1.min():.2f}, {layer.idx1.max():.2f}]')
print(f'{layer.y_label}: [{layer.idx2.min():.2f}, {layer.idx2.max():.2f}]')

fig, ax = plt.subplots(figsize=(8, 8))
s = layer.s
cx, cy = layer.cx, layer.cy
NX, NY = layer.intensity.shape[1], layer.intensity.shape[0]
extent = [(0.5-cx)*s, (NX+0.5-cx)*s, (0.5-cy)*s, (NY+0.5-cy)*s]

disp = layer.intensity.astype(float)
if layer.idx2[-1, NX//2] < layer.idx2[0, NX//2]:
    disp = disp[::-1, :]

nz = disp[disp > 0]
vmax = np.percentile(nz, 99) if len(nz) > 0 else 1
ax.imshow(disp, extent=extent, origin='lower', aspect='equal',
          cmap='gnuplot2', norm=Normalize(vmin=0, vmax=vmax))
ax.set_xlabel(layer.x_label); ax.set_ylabel(layer.y_label)
ax.set_title(f'{layer.fixed_label} = {layer.fixed_value:.3f}');

## 7. Launch Interactive GUI

Launch the unified viewer from the notebook for interactive exploration.

In [ ]:
%matplotlib qt
from rsp_unified_viewer import UnifiedViewer
from PyQt6.QtWidgets import QApplication
import sys

app = QApplication.instance() or QApplication(sys.argv)
viewer = UnifiedViewer()
viewer.show()

# Load a file programmatically:
# viewer._load_file('path/to/volume.h5')
# viewer._load_file('path/to/single.img')